# Light-Front QFT Simulation on a Gate-Based Quantum Computer

Consider a (1+1)-dimensional Yukawa model — a fermion and antifermion field coupled to a bosonic field — formulated on the **light front**. Truncating each momentum mode to a small Fock space yields a finite Hamiltonian that can be written as a sum of Pauli strings on 12 qubits. Starting from a single fermion in mode 2, $|f\rangle_2$, the system evolves under this Hamiltonian and can transition into the state $|f\rangle_1 \otimes |\phi\rangle_1$, a fermion in mode 1 together with one pion in mode 1.

Classically, the exact evolution $|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$ is tractable here (Hilbert-space dimension $2^{12} = 4096$), but the same Pauli decomposition is exactly what a gate-based quantum computer needs for **Hamiltonian simulation**. This notebook follows Section IV.1 of Vinod & Shaji [[1](#VinodShaji)]: we build the light-front Hamiltonian $H = H_M + H_V + H_S + H_F$, compare exact evolution with a first-order Suzuki–Trotter circuit synthesized through Classiq, and plot the population of $|f\rangle_1 \otimes |\phi\rangle_1$ as a function of time — reproducing **Figure 2** of the paper.

In [ ]:
import itertools
import math
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import expm_multiply

import classiq
from classiq import (
    CReal,
    IndexedPauli,
    Output,
    Pauli,
    QArray,
    QBit,
    SparsePauliOp,
    SparsePauliTerm,
    X,
    allocate,
    create_model,
    qfunc,
    show,
    suzuki_trotter,
    synthesize,
    write_qmod,
)
from classiq.execution import (
    ClassiqBackendPreferences,
    ClassiqSimulatorBackendNames,
    ExecutionPreferences,
    ExecutionSession,
)

RNG_SEED = 1234
np.random.seed(RNG_SEED)

print(f"classiq     {classiq.__version__}")
print(f"numpy       {np.__version__}")
print(f"scipy       {__import__('scipy').__version__}")
print(f"matplotlib  {__import__('matplotlib').__version__}")

## Quantum Algorithm

The light-front Hamiltonian of Section IV.1 decomposes into four terms at successive orders in the coupling $g = \lambda / \sqrt{4\pi}$:

* **$H_M$** — renormalized mass terms (number operators for fermions, antifermions, and bosons);
* **$H_V$** — $O(g)$ vertex interactions (emission/absorption and pair creation);
* **$H_S$** — $O(g^2)$ seagull (instantaneous two-boson exchange);
* **$H_F$** — $O(g^2)$ fork (one particle splitting into three, and reverse).

Fermionic modes use Jordan–Wigner encoding with $Z$-strings; bosonic modes use a compact binary encoding of occupancies $0\ldots 3$. Each term is built as a dictionary of 12-character Pauli strings, then converted to a sparse matrix for the classical reference evolution.

For the gate-based simulation we approximate

$$U(t) = e^{-iHt} \approx \left[\prod_j e^{-i h_j P_j\, t / n_T}\right]^{n_T}$$

using Classiq's `suzuki_trotter` with $n_T = 10$ first-order steps (the paper's Section IV.1 setting). The evolution time $t$ is a classical execution parameter, so the circuit is synthesized once and sampled at many times in a single `ExecutionSession`.

We track the population of the target Fock state $|100\,000\,01\,00\,00\rangle$ starting from $|010\,000\,00\,00\,00\rangle$.

In [ ]:
# --- Section IV.1 parameters (arXiv:2401.04496) ------------------------------
N_MAX = 3  # number of momentum modes per particle species (N_F = N_A = N_B)
N_MODALS = 3  # bosonic modal cutoff m (occupancies 0..3)
LAMBDA_COUPLING = 4.0  # bare Yukawa coupling lambda
M_F = 6.7  # fermion (proton) mass, in pion-mass units
M_B = 1.0  # boson (pion) mass, in pion-mass units
LAMBDA_CUTOFF = 2048  # harmonic-resolution cutoff in the inertia sums

G = LAMBDA_COUPLING / math.sqrt(4 * math.pi)  # g = lambda / sqrt(4 pi)

# --- Qubit register layout ----------------------------------------------------
QUBITS_PER_BOSON = math.ceil(math.log2(N_MODALS + 1))  # = 2
N_QUBITS = 2 * N_MAX + QUBITS_PER_BOSON * N_MAX  # 3 + 3 + 6 = 12
DIM = 2**N_QUBITS  # 4096

# --- States of the Section IV.1 experiment ------------------------------------
INITIAL_STATE = "010 000 00 00 00"  # |f>_2 : one fermion in mode 2   (K=2, Q=1)
TARGET_STATE = "100 000 01 00 00"  # |f>_1 (x) |phi>_1               (K=2, Q=1)

print(f"g = {G:.6f}")
print(f"qubits: {N_QUBITS}  (Hilbert-space dimension {DIM})")
# --- Pauli-string algebra ------------------------------------------------------
# An operator is a dict: {12-char Pauli string over "IXYZ" -> complex coefficient}.

_PAULI_MUL = {
    ("I", "I"): ("I", 1), ("I", "X"): ("X", 1), ("I", "Y"): ("Y", 1), ("I", "Z"): ("Z", 1),
    ("X", "I"): ("X", 1), ("X", "X"): ("I", 1), ("X", "Y"): ("Z", 1j), ("X", "Z"): ("Y", -1j),
    ("Y", "I"): ("Y", 1), ("Y", "X"): ("Z", -1j), ("Y", "Y"): ("I", 1), ("Y", "Z"): ("X", 1j),
    ("Z", "I"): ("Z", 1), ("Z", "X"): ("Y", 1j), ("Z", "Y"): ("X", -1j), ("Z", "Z"): ("I", 1),
}
_TOL = 1e-12  # drop terms with |coefficient| below this


def op_add(*ops):
    """Sum of Pauli-string operators, dropping numerically-zero terms."""
    out = {}
    for op in ops:
        for string, coeff in op.items():
            out[string] = out.get(string, 0) + coeff
    return {s: c for s, c in out.items() if abs(c) > _TOL}


def op_scale(op, factor):
    """Scalar multiple of a Pauli-string operator."""
    return {s: c * factor for s, c in op.items() if abs(c * factor) > _TOL}


def op_mul(op_a, op_b):
    """Operator product, using the single-qubit Pauli multiplication table."""
    out = {}
    for s_a, c_a in op_a.items():
        for s_b, c_b in op_b.items():
            phase = c_a * c_b
            chars = []
            for p_a, p_b in zip(s_a, s_b):
                pauli, extra_phase = _PAULI_MUL[(p_a, p_b)]
                chars.append(pauli)
                phase *= extra_phase
            string = "".join(chars)
            out[string] = out.get(string, 0) + phase
    return {s: c for s, c in out.items() if abs(c) > _TOL}


def op_product(*ops):
    """Left-to-right product of several operators (ordering matters for fermions)."""
    result = ops[0]
    for op in ops[1:]:
        result = op_mul(result, op)
    return result


def op_adjoint(op):
    """Hermitian conjugate (Pauli strings are self-adjoint; conjugate coefficients)."""
    return {s: np.conj(c) for s, c in op.items()}


def embed(single_qubit_op, qubit, z_qubits=()):
    """Embed a single-qubit operator at `qubit`, with Z on `z_qubits` (JW strings)."""
    out = {}
    for char, coeff in single_qubit_op.items():
        chars = ["I"] * N_QUBITS
        for q in z_qubits:
            chars[q] = "Z"
        chars[qubit] = char
        out["".join(chars)] = coeff
    return out


# Single-qubit building blocks, Eq. (9) of the paper.
SIGMA_PLUS = {"X": 0.5, "Y": 0.5j}  # |0><1|
SIGMA_MINUS = {"X": 0.5, "Y": -0.5j}  # |1><0|
I_PLUS = {"I": 0.5, "Z": 0.5}  # |0><0|
I_MINUS = {"I": 0.5, "Z": -0.5}  # |1><1|
def _fermionic_creation(jw_index):
    """Jordan-Wigner creation operator for JW mode `jw_index` (0..5)."""
    return embed(SIGMA_MINUS, jw_index, z_qubits=tuple(range(jw_index)))


def b_dag(n):
    """Fermion creation operator b_n^dag (mode n = 1..N_MAX, qubit n-1)."""
    return _fermionic_creation(n - 1)


def b_op(n):
    """Fermion annihilation operator b_n."""
    return op_adjoint(b_dag(n))


def d_dag(n):
    """Antifermion creation operator d_n^dag (mode n = 1..N_MAX, qubit N_MAX+n-1)."""
    return _fermionic_creation(N_MAX + n - 1)


def d_op(n):
    """Antifermion annihilation operator d_n."""
    return op_adjoint(d_dag(n))


def a_dag(n):
    """Truncated bosonic creation operator a_n^dag, Eq. (10) with m = 3 modals.

    Acts on the two qubits of bosonic mode n (first qubit = MSB of the occupancy):
    a^dag = sqrt(1) I+ (x) sigma-  +  sqrt(2) sigma- (x) sigma+  +  sqrt(3) I- (x) sigma-.
    """
    msb = 2 * N_MAX + QUBITS_PER_BOSON * (n - 1)
    lsb = msb + 1
    return op_add(
        op_scale(op_mul(embed(I_PLUS, msb), embed(SIGMA_MINUS, lsb)), math.sqrt(1)),
        op_scale(op_mul(embed(SIGMA_MINUS, msb), embed(SIGMA_PLUS, lsb)), math.sqrt(2)),
        op_scale(op_mul(embed(I_MINUS, msb), embed(SIGMA_MINUS, lsb)), math.sqrt(3)),
    )


def a_op(n):
    """Truncated bosonic annihilation operator a_n."""
    return op_adjoint(a_dag(n))


def c_dag(n):
    """Rescaled bosonic creation operator c_n^dag = a_n^dag / sqrt(n)."""
    return op_scale(a_dag(n), 1 / math.sqrt(n))


def c_op(n):
    """Rescaled bosonic annihilation operator c_n = a_n / sqrt(n)."""
    return op_scale(a_op(n), 1 / math.sqrt(n))
def nm_bracket(n, m):
    """The bracket {n|m} of Eq. (A.2): 1/n if m == -n != 0, else 0."""
    if n != 0 and m == -n:
        return 1.0 / n
    return 0.0


def alpha_bare(n):
    """Bare bosonic self-induced inertia alpha_n (Appendix), cutoff LAMBDA_CUTOFF."""
    return sum(
        nm_bracket(n - m, m - n) - nm_bracket(n + m, -m - n)
        for m in range(1, LAMBDA_CUTOFF + 1)
    )


def alpha(n):
    """Renormalized alpha_n (authors' reference-implementation convention)."""
    if n == 1:
        return -alpha_bare(1)
    return alpha_bare(n) - alpha_bare(1)


def beta(n):
    """Fermionic self-induced inertia beta_n (Appendix), cutoff LAMBDA_CUTOFF."""
    return sum(
        (n / m) * nm_bracket(n - m, m - n) for m in range(1, LAMBDA_CUTOFF + 1)
    )


def gamma(n):
    """Antifermionic self-induced inertia gamma_n (Appendix), cutoff LAMBDA_CUTOFF."""
    return sum(
        (n / m) * nm_bracket(n + m, -m - n) for m in range(1, LAMBDA_CUTOFF + 1)
    )


print("n | alpha_bare(n) | alpha(n) | beta(n) | gamma(n)")
for n in range(1, N_MAX + 1):
    print(
        f"{n} | {alpha_bare(n):13.4f} | {alpha(n):8.4f}"
        f" | {beta(n):7.4f} | {gamma(n):8.4f}"
    )
def build_h_m():
    """Mass term H_M: diagonal number operators with renormalized masses."""
    terms = []
    for n in range(1, N_MAX + 1):
        terms.append(
            op_scale(op_mul(a_dag(n), a_op(n)), (M_B**2 + G**2 * alpha(n)) / n)
        )
        terms.append(
            op_scale(op_mul(b_dag(n), b_op(n)), (M_F**2 + G**2 * beta(n)) / n)
        )
        terms.append(
            op_scale(op_mul(d_dag(n), d_op(n)), (M_F**2 + G**2 * gamma(n)) / n)
        )
    return op_add(*terms)


def build_h_v():
    """Vertex term H_V: O(g) boson emission/absorption and pair creation."""
    terms = []
    for k, l, m in itertools.product(range(1, N_MAX + 1), repeat=3):
        cf_emit = nm_bracket(k + l, -m) + nm_bracket(k, l - m)
        cf_pair = nm_bracket(k - l, m) + nm_bracket(k, -l + m)
        if cf_emit:
            terms.append(op_scale(op_add(
                op_product(b_dag(k), b_op(m), c_dag(l)),
                op_product(b_dag(m), b_op(k), c_op(l)),
                op_product(d_dag(k), d_op(m), c_dag(l)),
                op_product(d_dag(m), d_op(k), c_op(l)),
            ), cf_emit))
        if cf_pair:
            terms.append(op_scale(op_add(
                op_product(b_op(k), d_op(m), c_dag(l)),
                op_product(d_dag(m), b_dag(k), c_op(l)),
            ), -cf_pair))
    return op_scale(op_add(*terms), G * M_F)


def build_h_s():
    """Seagull term H_S: O(g^2) instantaneous two-boson exchange."""
    terms = []
    for k, l, m, n in itertools.product(range(1, N_MAX + 1), repeat=4):
        cf_scatter = nm_bracket(k - n, l - m) + nm_bracket(k + l, -m - n)
        cf_pair = nm_bracket(l - k, n - m)
        if cf_scatter:
            terms.append(op_scale(op_add(
                op_product(b_dag(k), b_op(m), c_dag(l), c_op(n)),
                op_product(d_dag(k), d_op(m), c_dag(l), c_op(n)),
            ), cf_scatter))
        if cf_pair:
            terms.append(op_scale(op_add(
                op_product(d_op(k), b_op(m), c_dag(l), c_dag(n)),
                op_product(b_dag(m), d_dag(k), c_op(n), c_op(l)),
            ), cf_pair))
    return op_scale(op_add(*terms), G**2)


def build_h_f():
    """Fork term H_F: O(g^2) one particle forking into three (and reverse)."""
    terms = []
    for k, l, m, n in itertools.product(range(1, N_MAX + 1), repeat=4):
        cf_fork = nm_bracket(k + l, n - m)
        cf_mixed = nm_bracket(k - n, m + l) + nm_bracket(k + l, m - n)
        if cf_fork:
            terms.append(op_scale(op_add(
                op_product(b_dag(k), b_op(m), c_dag(l), c_dag(n)),
                op_product(b_dag(m), b_op(k), c_op(n), c_op(l)),
                op_product(d_dag(k), d_op(m), c_dag(l), c_dag(n)),
                op_product(d_dag(m), d_op(k), c_op(n), c_op(l)),
            ), cf_fork))
        if cf_mixed:
            terms.append(op_scale(op_add(
                op_product(b_dag(k), d_dag(m), c_dag(l), c_op(n)),
                op_product(d_op(m), b_op(k), c_dag(n), c_op(l)),
            ), cf_mixed))
    return op_scale(op_add(*terms), G**2)


h_m = build_h_m()
h_v = build_h_v()
h_s = build_h_s()
h_f = build_h_f()
h_pauli = op_add(h_m, h_v, h_s, h_f)

print("Pauli-string counts:")
print(f"  H_M: {len(h_m):5d}")
print(f"  H_V: {len(h_v):5d}")
print(f"  H_S: {len(h_s):5d}")
print(f"  H_F: {len(h_f):5d}")
print(f"  H  : {len(h_pauli):5d}  (after combining identical strings)")

weights = Counter(sum(1 for ch in s if ch != "I") for s in h_pauli)
print("\nPauli-weight histogram (weight: count):")
print(" ", dict(sorted(weights.items())))
def pauli_string_to_sparse(string, coeff):
    """Sparse (COO) matrix of one weighted Pauli string.

    Qubit 0 is the leftmost character and the most-significant bit, matching the
    paper's ket notation. A Pauli string acts on basis state |x> as a bit-flip
    (X, Y characters) times a phase from the Y and Z characters.
    """
    x_mask = y_mask = z_mask = 0
    for q, char in enumerate(string):
        bit = 1 << (N_QUBITS - 1 - q)
        if char == "X":
            x_mask |= bit
        elif char == "Y":
            x_mask |= bit
            y_mask |= bit
        elif char == "Z":
            z_mask |= bit

    cols = np.arange(DIM, dtype=np.int64)
    rows = cols ^ x_mask  # X/Y characters flip bits

    # Phase: i^(#Y) times (-1)^(parity of input bits under the Y+Z masks).
    parity = np.zeros(DIM, dtype=np.int64)
    masked = cols & (y_mask | z_mask)
    for _ in range(N_QUBITS):
        parity ^= masked & 1
        masked >>= 1
    n_y = bin(y_mask).count("1")
    values = coeff * (1j**n_y) * np.where(parity, -1.0, 1.0)
    return sp.coo_matrix((values, (rows, cols)), shape=(DIM, DIM))


def pauli_op_to_sparse(op):
    """Sparse CSR matrix of a Pauli-string operator dict."""
    total = sp.coo_matrix((DIM, DIM), dtype=complex)
    for string, coeff in op.items():
        total = total + pauli_string_to_sparse(string, coeff)
    return total.tocsr()


def ket_index(bitstring):
    """Basis index of a paper-notation ket like '010 000 00 00 00'."""
    return int(bitstring.replace(" ", ""), 2)


h_matrix = pauli_op_to_sparse(h_pauli)
idx_initial = ket_index(INITIAL_STATE)
idx_target = ket_index(TARGET_STATE)

print(f"H matrix: {h_matrix.shape}, nnz = {h_matrix.nnz}"
      f" (density {h_matrix.nnz / DIM**2:.2%})")
print(f"initial state |f>_2          -> index {idx_initial}")
print(f"target  state |f>_1|phi>_1   -> index {idx_target}")

## Exact Classical Reference

The exact evolution uses SciPy's `expm_multiply` on the sparse Hamiltonian. Section IV.1 uses a uniform time grid $t \in [0, 1]$ with step $0.01$. Because the initial and target states both carry the same conserved charges $K = 2$ and $Q = 1$, the dynamics are well approximated by a two-level Rabi oscillation between $|f\rangle_2$ and $|f\rangle_1 \otimes |\phi\rangle_1$ — a useful cross-check on the matrix elements extracted from $H$.

In [ ]:
e_initial = h_matrix[idx_initial, idx_initial].real
e_target = h_matrix[idx_target, idx_target].real
v_coupling = h_matrix[idx_initial, idx_target].real

delta = e_target - e_initial
omega = math.sqrt(v_coupling**2 + delta**2 / 4)
rabi_amplitude = v_coupling**2 / omega**2
rabi_period = math.pi / omega
e_mean = (e_initial + e_target) / 2

print(f"E_i (|f>_2)          = {e_initial:9.4f}")
print(f"E_f (|f>_1|phi>_1)   = {e_target:9.4f}")
print(f"V   (coupling)       = {v_coupling:9.4f}")
print(f"Omega                = {omega:9.4f}")
print(f"eigenvalues E+/-     = {e_mean + omega:9.4f}, {e_mean - omega:9.4f}")
print(f"predicted amplitude  = {rabi_amplitude:9.4f}")
print(f"predicted period     = {rabi_period:9.4f}  (m_pi^-1)")


def rabi_population(t):
    """Closed-form two-level transition probability P(t) = (V/Omega)^2 sin^2(Omega t)."""
    return rabi_amplitude * np.sin(omega * np.asarray(t)) ** 2
# Conserved-charge operators (Eqs. (6)-(7)), reused in validation below.
k_pauli, q_pauli = {}, {}
for n in range(1, N_MAX + 1):
    k_pauli = op_add(
        k_pauli,
        op_scale(op_mul(a_dag(n), a_op(n)), n),
        op_scale(op_mul(b_dag(n), b_op(n)), n),
        op_scale(op_mul(d_dag(n), d_op(n)), n),
    )
    q_pauli = op_add(
        q_pauli,
        op_mul(b_dag(n), b_op(n)),
        op_scale(op_mul(d_dag(n), d_op(n)), -1),
    )
k_matrix = pauli_op_to_sparse(k_pauli)
q_matrix = pauli_op_to_sparse(q_pauli)

T_MAX = 1.0  # total evolution time, units of m_pi^-1
N_EXACT = 101  # paper Section IV.1: step 0.01 over [0, 1]

times_exact = np.linspace(0.0, T_MAX, N_EXACT)
psi_0 = np.zeros(DIM, dtype=complex)
psi_0[idx_initial] = 1.0

# expm_multiply applies e^{-iHt} to psi_0 on the whole uniform grid at once.
states = expm_multiply(-1j * h_matrix, psi_0, start=0.0, stop=T_MAX, num=N_EXACT)

populations_exact = np.abs(states[:, idx_target]) ** 2
survival_exact = np.abs(states[:, idx_initial]) ** 2
norms_exact = np.sum(np.abs(states) ** 2, axis=1)
# Expectation values via sparse mat-vecs (K and Q are real symmetric).
k_expect = np.real(np.einsum("ti,ti->t", states.conj(), (k_matrix @ states.T).T))
q_expect = np.real(np.einsum("ti,ti->t", states.conj(), (q_matrix @ states.T).T))

print(f"max |1 - <psi|psi>|          : {np.abs(1 - norms_exact).max():.2e}")
print(f"<K> range                    : [{k_expect.min():.12f}, {k_expect.max():.12f}]")
print(f"<Q> range                    : [{q_expect.min():.12f}, {q_expect.max():.12f}]")
print(f"max leakage out of 2D space  : "
      f"{np.max(1 - populations_exact - survival_exact):.2e}")
print(f"peak transition probability  : {populations_exact.max():.4f}")
print(f"P(t = 0.2)                   : "
      f"{populations_exact[np.argmin(np.abs(times_exact - 0.2))]:.4f}")
def style_paper_time_axes(ax, ylabel):
    """Axis styling shared with arXiv:2401.04496 Figs. 2-3."""
    ax.set_xlim(0, T_MAX)
    ax.set_xticks(np.arange(0, T_MAX + 1e-9, 0.2))
    ax.tick_params(direction="in", top=True, right=True)
    ax.set_xlabel(r"evolution time (in $m_{\pi}^{-1}$)", fontweight="bold")
    ax.set_ylabel(ylabel, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)


PAPER_WINDOW = "#b8e0b8"  # light green band used in the paper's Fig. 3

We are now ready to run our main execution function, `run_qft_evolution`. This function synthesizes a single parametric circuit that performs Hamiltonian simulation $e^{-iHt}$ using `suzuki_trotter`. The evolution time `t` is declared as a classical execution parameter (`CReal`), so the circuit is synthesized only once and then sampled at many time values in a single `ExecutionSession`.

The initial Fock state $|f\rangle_2$ is prepared by flipping the occupied qubits of $|010\,000\,00\,00\,00\rangle$. We sample with 8192 shots per time point (matching the paper) and read out the population of $|f\rangle_1 \otimes |\phi\rangle_1$.

> **Bit-ordering caveat.** Paper kets use **qubit 0 as the leftmost character**; Classiq measurement keys use **qubit 0 as the rightmost character**. The helper `population_from_counts` reverses the paper-notation bitstring before lookup.

In [ ]:
PAULI_ENUM = {"X": Pauli.X, "Y": Pauli.Y, "Z": Pauli.Z}


def pauli_op_to_sparse_pauli_op(op, decimals=4):
    """Convert a Pauli-string dict to a Classiq SparsePauliOp.

    Coefficients of a Hermitian Pauli decomposition are real; they are rounded
    to `decimals` places (the convention of the authors' reference code).
    """
    terms = []
    for string, coeff in sorted(op.items()):
        coefficient = round(float(np.real(coeff)), decimals)
        if coefficient == 0.0:
            continue
        indexed_paulis = [
            IndexedPauli(pauli=PAULI_ENUM[char], index=qubit)
            for qubit, char in enumerate(string)
            if char != "I"
        ]
        terms.append(
            SparsePauliTerm(paulis=indexed_paulis, coefficient=coefficient)
        )
    return SparsePauliOp(terms=terms, num_qubits=N_QUBITS)


hamiltonian_op = pauli_op_to_sparse_pauli_op(h_pauli)
max_imag = max(abs(np.imag(c)) for c in h_pauli.values())
print(f"SparsePauliOp with {len(hamiltonian_op.terms)} terms on "
      f"{hamiltonian_op.num_qubits} qubits")
print(f"max |imaginary part| of coefficients: {max_imag:.2e}  (Hermiticity)")
N_TROTTER = 10  # paper Section IV.1: <5% deviation at t = 0.2 with 10 steps
TROTTER_ORDER = 1  # paper's choice: first-order (second order costs 2x depth)

_INITIAL_BITS = INITIAL_STATE.replace(" ", "")
INITIAL_EXCITED_QUBITS = [q for q, bit in enumerate(_INITIAL_BITS) if bit == "1"]


@qfunc
def prepare_fock_state(state: QArray[QBit]):
    """Prepare the initial Fock state |f>_2 by flipping its occupied qubits.

    A Fock basis state is a computational basis state, so preparation is a
    layer of X gates on the occupied modes (here: qubit 1, fermion mode 2).
    """
    for qubit in INITIAL_EXCITED_QUBITS:
        X(state[qubit])


@qfunc
def main(t: CReal, state: Output[QArray[QBit]]):
    """Trotterized light-front evolution  |psi(t)> ~ [prod_j e^{-i h_j P_j t/nT}]^nT |f>_2.

    Args:
        t: evolution time (classical parameter, units of 1/m_pi).
        state: 12-qubit register holding the Fock state of the field theory.
    """
    allocate(N_QUBITS, state)
    prepare_fock_state(state)
    suzuki_trotter(
        hamiltonian_op,
        evolution_coefficient=t,
        order=TROTTER_ORDER,
        repetitions=N_TROTTER,
        qbv=state,
    )


model = create_model(main)
print("model created")
N_TROTTER = 10
TROTTER_ORDER = 1
NUM_SHOTS = 8192
TIME_STEP_SWEEP = 0.025

execution_preferences = ExecutionPreferences(
    num_shots=NUM_SHOTS,
    random_seed=RNG_SEED,
    backend_preferences=ClassiqBackendPreferences(
        backend_name=ClassiqSimulatorBackendNames.SIMULATOR_STATEVECTOR
    ),
)


def population_from_counts(counts, bitstring):
    """Measured population of a paper-notation basis state."""
    key = bitstring.replace(" ", "")[::-1]
    return counts.get(key, 0) / sum(counts.values())


def run_time_sweep(quantum_program, time_values):
    with ExecutionSession(quantum_program, execution_preferences) as session:
        results = session.sample([{"t": float(t)} for t in time_values])
    return [result.counts for result in results]


def run_qft_evolution(time_step=TIME_STEP_SWEEP, t_max=T_MAX, export_qmod=True):
    """Synthesize the Trotter circuit, optionally export .qmod, and sample a time sweep."""
    qprog = synthesize(model)
    show(qprog)

    transpiled = qprog.transpiled_circuit
    gate_counts = dict(transpiled.count_ops)
    circuit_depth = transpiled.depth
    print(f"circuit width : {qprog.data.width} qubits")
    print(f"circuit depth : {circuit_depth}")
    print(f"gate counts   : {gate_counts}")

    if export_qmod:
        write_qmod(model, "qft_simulation", decimal_precision=4)
        import os

        size_mb = os.path.getsize("qft_simulation.qmod") / 1e6
        print(f"qft_simulation.qmod written ({size_mb:.2f} MB)")

    times = np.round(np.arange(0.0, t_max + 1e-9, time_step), 6)
    counts_sweep = run_time_sweep(qprog, times)
    populations = np.array(
        [population_from_counts(c, TARGET_STATE) for c in counts_sweep]
    )
    return qprog, times, populations, counts_sweep, gate_counts, circuit_depth

The following cell runs `run_qft_evolution` with the Section IV.1 parameters ($n_T = 10$, $\Delta t = 0.025$ on $[0, 1]$). Synthesis of the 2257-term Hamiltonian takes a couple of minutes.

In [ ]:
(
    qprog,
    times_trotter,
    populations_trotter,
    counts_sweep,
    gate_counts,
    circuit_depth,
) = run_qft_evolution()

survival_trotter = np.array(
    [population_from_counts(c, INITIAL_STATE) for c in counts_sweep]
)
leak_trotter = 1.0 - populations_trotter - survival_trotter

print(f"{len(times_trotter)} time points sampled, {NUM_SHOTS} shots each")
print(
    f"P_target(t=0.2) = "
    f"{populations_trotter[np.argmin(np.abs(times_trotter - 0.2))]:.4f}"
)
print(f"max leakage out of the 2D subspace (Trotter): {leak_trotter.max():.4f}")

## Graph Results

The function below plots the exact reference curve together with the Classiq Trotter samples, reproducing Figure 2 of the paper. Error bars show $\sqrt{p(1-p)/N_{\mathrm{shots}}}$ shot noise.

In [ ]:
def graph_figure2(
    times_exact,
    populations_exact,
    times_trotter,
    populations_trotter,
    num_shots=NUM_SHOTS,
    n_trotter=N_TROTTER,
):
    shot_noise = np.sqrt(populations_trotter * (1 - populations_trotter) / num_shots)

    plt.rcParams.update({"font.size": 10, "axes.grid": False})
    fig, ax = plt.subplots(figsize=(5.8, 4.2))
    ax.axvspan(
        0,
        0.2,
        color=PAPER_WINDOW,
        alpha=0.95,
        zorder=0,
        label=r"validated window ($t \leq 0.2$, $n_T = 10$)",
    )
    ax.plot(
        times_exact,
        populations_exact,
        ":",
        color="C0",
        lw=1.3,
        zorder=2,
        label="exact evolution $e^{-iHt}$",
    )
    ax.plot(
        times_exact,
        populations_exact,
        "o",
        color="k",
        ms=2.8,
        markeredgewidth=0,
        zorder=3,
    )
    ax.errorbar(
        times_trotter,
        populations_trotter,
        yerr=shot_noise,
        fmt="o",
        ms=4.0,
        color="tab:red",
        ecolor="tab:red",
        capsize=2.0,
        lw=0.8,
        zorder=4,
        label=f"Classiq Suzuki-Trotter ($n_T$={n_trotter}, {num_shots} shots)",
    )
    style_paper_time_axes(
        ax,
        r"probability of the $|fangle_1 \otimes |\phiangle_1$ state",
    )
    ax.set_ylim(0, 0.25)
    ax.set_yticks(np.arange(0, 0.21, 0.05))
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()

Plot Figure 2: exact evolution vs. Classiq Trotter samples.

In [ ]:
graph_figure2(
    times_exact,
    populations_exact,
    times_trotter,
    populations_trotter,
)

## Circuit Resources

With 2257 Pauli terms and $n_T = 10$ Trotter steps, the transpiled circuit contains on the order of $10^4$ gates. The exported `qft_simulation.qmod` file can be inspected in the Classiq IDE or submitted to hardware backends once error mitigation and further Trotter refinement are applied.

In [ ]:
n_terms = len(hamiltonian_op.terms)
cx_count = gate_counts.get("cx", 0)
total_gates = sum(gate_counts.values())

print(f"logical qubits                : {qprog.data.width}")
print(f"Pauli terms in H              : {n_terms}")
print(f"Trotter repetitions           : {N_TROTTER}")
print(f"term-exponentials in circuit  : {n_terms * N_TROTTER}")
print(f"transpiled depth              : {circuit_depth}")
print(f"total gates                   : {total_gates}")
print(f"two-qubit (CX) gates          : {cx_count}")
print(f"CX per term-exponential       : {cx_count / (n_terms * N_TROTTER):.1f}")

## References

<a id="VinodShaji"></a>
[1] Gayathree M. Vinod and Anil Shaji, *Simulating Quantum Field Theories on Gate-Based Quantum Computers*, IEEE Transactions on Quantum Engineering **5**, 2500615 (2024). [arXiv:2401.04496](https://arxiv.org/abs/2401.04496), [DOI:10.1109/TQE.2024.3385372](https://doi.org/10.1109/TQE.2024.3385372).

[2] [Classiq Library issue #580](https://github.com/Classiq/classiq-library/issues/580) — contribution tracking this notebook.